In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from imblearn.over_sampling import SMOTE
from collections import Counter
import copy
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score
from sklearn.model_selection import cross_val_score, KFold, cross_validate
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import learning_curve
import matplotlib.pyplot as plt
from sklearn.ensemble import AdaBoostClassifier
import xgboost as xgb
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore")


%matplotlib inline

In [16]:
# Split datasets

def split_sets(X, y, test_size=0.2, random_state=42):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)
    y_train.replace({'No': 0, 'Yes': 1})
    y_test.replace({'No': 0, 'Yes': 1})
    print(f"Dimension of X_train: {X_train.shape}")
    print(f"Dimension of X_test: {X_test.shape}")
    print(f"Dimension of y_train: {y_train.shape}")
    print(f"Dimension of y_test: {y_test.shape}")
    return X_train, X_test, y_train, y_test

In [10]:
# Remove columns

# According to EDA, columns to remove: Transfusion History, Marital Status, Non Smoker (due to high correlation with 'Smoking History')

def remove_columns(df, columns_to_delete):
  for column in columns_to_delete:
    df = df.drop(columns=[column], errors='ignore')

In [11]:
# Remove unnecessary rows

def remove_rows(df):

  # 1. Remove duplicates
    duplicates_before = len(df)
    df = df.drop_duplicates().reset_index(drop=True)
    duplicates_removed = duplicates_before - len(df)
    print(f"# duplicates removed: {duplicates_removed}")

  # 2. Remove rows with missing values

    rows_before = len(df)
    df = df.dropna()
    rows_after = len(df)
    print(f"Number of rows deleted for having missing values {rows_before - rows_after}")

    return df

In [12]:
# Data imputation for categorical variables (with the mode)

def df_cat_imputation(df, values_to_imput_cat):
  mode_train={}

  for column in values_to_imput_cat.keys():
    mode_train[column] = df[df[column] != values_to_imput_cat[column]].loc[:, column].mode()[0]
    df.loc[df[column] == values_to_imput_cat[column], column] = mode_train[column]

  return df, mode_train

In [ ]:
# Standardize values in 'Urban or Real' column

def standardize_urbal_rural (df):
  df['Urban or Rural'] = df['Urban or Rural'].str.lower()
  df['Urban or Rural'] = df['Urban or Rural'].str.capitalize()

  return df

In [14]:
# Encode nominal variables into booleans

# Nominal columns (Yes/No): 'Diabetes History', 'Heart Disease History', 'Inflammatory Bowel Disease',
#        'Survival Prediction', 'Diabetes', 'Alcohol Consumption', 'Early Detection',
#        'Family History', 'Genetic Mutation'

def convert_into_bool(df, binary_cols):
  for col in binary_cols:
        if col in df.columns:
            df[col] = df[col].map({'Yes': 1, 'No': 0}).astype(int)

  return df

In [ ]:
# Transform 'Date of Birth' column into 'Age' column

def create_age_column(df, reference_date='2025-01-01'):


      if 'Date of Birth' in df.columns:
        # Create 'Age' Column
        try:
            df['Date of Birth'] = pd.to_datetime(df['Date of Birth'], errors='coerce')
            ref_date = pd.Timestamp(reference_date)
            df['Age'] = ((ref_date - df['Date of Birth']).dt.days / 365.25).round()
            df['Age'] = df['Age'].astype('float')

            # Delete 'Date of birth'
            df = df.drop(columns=['Date of Birth'])
            print(f"'Date of Birth' transformed into 'Age'")
        except Exception as e:
            print(f"There was an error {e}")

In [15]:
# Preprocessing for numerical variables (winsorization for outliers, imputation with median)

# Numerical columns to preprocess here: 'Healthcare Costs', 'Incidence Rate per 100K', 'Mortality Rate per 100K',
#        'Tumor Size (mm)'

# Column 'Healthcare Costs' has negative values. These will be imputated with the median.

def df_num_imputation(df, numeric_cols):

  stats_pre = {}

# Transform columns into float type
  for col in numeric_cols:
    if col in df.columns:
      df[col] = pd.to_numeric(df[col], errors='coerce')

# Remove outliers using 'Winsorization'
  for column in numeric_cols:
      Q1 = df[column].quantile(0.25)
      Q3 = df[column].quantile(0.75)
      IQR = Q3 - Q1

      lower_bound = Q1 - 1.5 * IQR
      upper_bound = Q3 + 1.5 * IQR

      # Clip outliers with upper or lower bound
      df[column] = df[column].clip(lower=lower_bound, upper=upper_bound)

      # Imputing negative values with the median
      median_c = df[column].median()
      df.loc[df[column] < 0, column] = median_c
      stats_pre[column] = {
          'lower_bound': lower_bound,
          'upper_bound': upper_bound,
          'median': median_c
      }

  # Returning the df and the stats_pre dictionary which will be used later to preprocess the test set
  return df, stats_pre

In [ ]:
# Create dummy columns for categorical ordinal variables

# Those categoricals with less than 2 unique values will be label encoded.

# The main idea is to get rid of categorical columns by transforming them into dummies

# Columns to process here
#'Cancer Stage', 'Country', 'Diet Risk', 'Gender', 'Healthcare Access',
#        'Insurance Costs', 'Insurance Status', 'Obesity BMI', 'Physical Activity',
#        'Screening History', 'Smoking History', 'Treatment Type', 'Urban or Rural'

def create_dummies(df, categorical_cols):
  for col in categorical_cols:
    if col in df.columns:
      unique_values = df[col].nunique()
      if unique_values > 2:  # One-hot encoding will be performed on variables with more than 2 unique values
        dummies = pd.get_dummies(df[col], prefix=col, drop_first=False).astype(int)
        df = pd.concat([df, dummies], axis=1)
        df = df.drop(columns=[col])  # Drop original column after getting dummies

  # Identify remaining categorical columns

  non_numeric_cols = []
  for col in df.columns:
      if df[col].dtype == 'object':
          print(f"Non numeric column found: {col}")
          non_numeric_cols.append(col)

  # Transform previous columns

  for col in non_numeric_cols:
    # Customized code for 'Gender' column
    if col in df.columns:
      if col == 'Gender' and 'Gender' in df.columns:
        df['Gender'] = df['Gender'].map({'M': 0, 'F': 1}).astype(int)
      else:
        dummies = pd.get_dummies(df[col], prefix=col, drop_first=True)
        df = pd.concat([df, dummies], axis=1)
        df = df.drop(columns=[col])

  # Check for non-numeric remaining columns

  for col in df.columns:
    if df[col].dtype == 'object':
      print(f"Column {col} remains as non-numeric")

  return df

In [ ]:
# Macro function for gathering the previous ones

def preprocess_cancer_df(X, y, columns_to_delete, values_to_imput_cat, binary_cols, numeric_cols, categorical_cols):
